In [ ]:
from google.colab import files
files.upload()  # upload your kaggle.json


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"naveen1503","key":"2c6851ec360880cd33362409e462f1ca"}'}

In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d chethuhn/network-intrusion-dataset  # example
!unzip network-intrusion-dataset.zip -d /content/cicids2017


Dataset URL: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
License(s): CC0-1.0
 83% 190M/230M [00:00<00:00, 415MB/s] 
100% 230M/230M [00:00<00:00, 490MB/s]
Archive:  network-intrusion-dataset.zip
  inflating: /content/cicids2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  
  inflating: /content/cicids2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  
  inflating: /content/cicids2017/Friday-WorkingHours-Morning.pcap_ISCX.csv  
  inflating: /content/cicids2017/Monday-WorkingHours.pcap_ISCX.csv  
  inflating: /content/cicids2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  
  inflating: /content/cicids2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  
  inflating: /content/cicids2017/Tuesday-WorkingHours.pcap_ISCX.csv  
  inflating: /content/cicids2017/Wednesday-workingHours.pcap_ISCX.csv  


In [ ]:
import pandas as pd
df = pd.read_csv("/content/cicids2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")


In [ ]:
print(df.columns.tolist())


[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [ ]:
df.columns = df.columns.str.strip()
X = df.drop('Label', axis=1)
y = df['Label']

In [ ]:
import numpy as np
# 2️⃣ Clean column names
df.columns = df.columns.str.strip()

# 3️⃣ Replace inf and -inf with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# 4️⃣ Drop rows with NaN or very large outliers
df = df.dropna()
df = df[(df != np.inf).all(axis=1)]

# 5️⃣ Encode labels
df['Label'] = df['Label'].replace({'BENIGN': 0})
df['Label'] = df['Label'].apply(lambda x: 1 if x != 0 else 0)

# 6️⃣ Separate features and target
X = df.drop('Label', axis=1)
y = df['Label']

# 7️⃣ Convert all columns to numeric (sometimes strings sneak in)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# 8️⃣ Normalize
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 9️⃣ Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("✅ Clean data ready for SNN training")
print("Training samples:", X_train.shape[0])

✅ Clean data ready for SNN training
Training samples: 180568


In [ ]:
df['Label'].unique()
df['Label'].value_counts()


,count
Label,
1,128025
0,97686


In [ ]:
df.dtypes

,0
Destination Port,int64
Flow Duration,int64
Total Fwd Packets,int64
Total Backward Packets,int64
Total Length of Fwd Packets,int64
...,...
Idle Mean,float64
Idle Std,float64
Idle Max,int64
Idle Min,int64


In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from snntorch.functional import loss as snn_loss

# Convert numpy arrays to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

# Define SNN model
beta = 0.95  # membrane potential decay

class SNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 256)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(256, 2)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk1, mem1 = self.lif1(self.fc1(x), mem1)
        spk2, mem2 = self.lif2(self.fc2(spk1), mem2)
        return spk2, mem2

model = SNNModel()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
n_epochs = 10
for epoch in range(n_epochs):
    optimizer.zero_grad()
    spk_out, mem_out = model(X_train_t)
    loss = criterion(mem_out, y_train_t)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.6930
Epoch 2, Loss: 0.6929
Epoch 3, Loss: 0.6928
Epoch 4, Loss: 0.6926
Epoch 5, Loss: 0.6925
Epoch 6, Loss: 0.6924
Epoch 7, Loss: 0.6923
Epoch 8, Loss: 0.6921
Epoch 9, Loss: 0.6920
Epoch 10, Loss: 0.6919


In [ ]:
!pip install snntorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 3.2 MB/s eta 0:00:00


In [ ]:
with torch.no_grad():
    spk_out, mem_out = model(X_test_t)
    preds = torch.argmax(mem_out, 1)
    acc = (preds == y_test_t).float().mean()
    print(f"Test Accuracy: {acc.item()*100:.2f}%")


Test Accuracy: 56.98%


In [ ]:
from sklearn.utils import resample

# Separate classes
df_majority = df[df['Label'] == 0]
df_minority = df[df['Label'] == 1]

# Determine the target size
n_samples = min(len(df_majority), len(df_minority))

# Downsample both classes to match n_samples
df_majority_down = resample(df_majority, replace=False, n_samples=n_samples, random_state=42)
df_minority_down = resample(df_minority, replace=False, n_samples=n_samples, random_state=42)

# Combine
df_balanced = pd.concat([df_majority_down, df_minority_down])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced['Label'].value_counts())


Label
1    97686
0    97686
Name: count, dtype: int64


In [ ]:
X = df_balanced.drop('Label', axis=1)
y = df_balanced['Label']
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


In [ ]:
import torch

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)


In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
import torch.nn as nn

criterion = nn.CrossEntropyLoss()  # works for multi-class or binary classification

from torch.utils.data import TensorDataset, DataLoader

In [ ]:
batch_size = 128
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
beta = 0.95  # membrane decay

class SNNModel(nn.Module):
    def __init__(self, input_size, hidden1=512, hidden2=128, output_size=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.lif2 = snn.Leaky(beta=beta)
        self.fc3 = nn.Linear(hidden2, output_size)
        self.lif3 = snn.Leaky(beta=beta)

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        spk1, mem1 = self.lif1(self.fc1(x), mem1)
        spk2, mem2 = self.lif2(self.fc2(spk1), mem2)
        spk3, mem3 = self.lif3(self.fc3(spk2), mem3)
        return mem3  # final membrane potential for classification


In [ ]:
input_size = X_train.shape[1]
model = SNNModel(input_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()


In [ ]:
n_epochs = 30

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {epoch_loss/len(train_loader):.4f}")


Epoch 1/30, Loss: 0.5034
Epoch 2/30, Loss: 0.1979
Epoch 3/30, Loss: 0.0337
Epoch 4/30, Loss: 0.0272
Epoch 5/30, Loss: 0.0243
Epoch 6/30, Loss: 0.0198
Epoch 7/30, Loss: 0.0190
Epoch 8/30, Loss: 0.0183
Epoch 9/30, Loss: 0.0175
Epoch 10/30, Loss: 0.0172
Epoch 11/30, Loss: 0.0158
Epoch 12/30, Loss: 0.0138
Epoch 13/30, Loss: 0.0134
Epoch 14/30, Loss: 0.0128
Epoch 15/30, Loss: 0.0125
Epoch 16/30, Loss: 0.0119
Epoch 17/30, Loss: 0.0113
Epoch 18/30, Loss: 0.0090
Epoch 19/30, Loss: 0.0045
Epoch 20/30, Loss: 0.0040
Epoch 21/30, Loss: 0.0035
Epoch 22/30, Loss: 0.0035
Epoch 23/30, Loss: 0.0034
Epoch 24/30, Loss: 0.0038
Epoch 25/30, Loss: 0.0041
Epoch 26/30, Loss: 0.0039
Epoch 27/30, Loss: 0.0038
Epoch 28/30, Loss: 0.0030
Epoch 29/30, Loss: 0.0041
Epoch 30/30, Loss: 0.0036


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        out = model(X_batch)
        preds = torch.argmax(out, dim=1)
        all_preds.append(preds)

y_pred = torch.cat(all_preds)
print(classification_report(y_test_t, y_pred))
print(confusion_matrix(y_test_t, y_pred))
print(accuracy_score(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19630
           1       1.00      1.00      1.00     19445

    accuracy                           1.00     39075
   macro avg       1.00      1.00      1.00     39075
weighted avg       1.00      1.00      1.00     39075

[[19626     4]
 [   15 19430]]
0.9995137555982085
